# Task 1: LoRA Fine-tuning on TrialGPT Dataset (Colab T4)

This notebook reproduces the complete Task 1 pipeline:
1. Download & fixed patient-grouped split (37/8/8, seed 42)
2. LoRA fine-tune Qwen2.5-1.5B-Instruct on training split
3. Save adapter + training logs
4. (Optional) Run before/after evaluation on validation split

**Runtime**: ~15-30 min on free Colab T4 (16 GB VRAM)
**Output**: `adapters/qwen25-1.5b-lora/` + `results/training_log.json` + `results/train_curve.csv`

In [ ]:
# @title 1. Install dependencies
%pip install -q accelerate peft transformers datasets scikit-learn pandas numpy tqdm

# Verify GPU
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# @title 2. Clone repository (replace with your GitHub URL)
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # <-- EDIT THIS

import subprocess, os
if not os.path.exists("genAI_A2"):
    subprocess.run(["git", "clone", REPO_URL, "genAI_A2"], check=True)
os.chdir("genAI_A2")
print(f"Working in: {os.getcwd()}")
!ls -la

In [ ]:
# @title 3. Prepare dataset & fixed split (runs prepare_data.py)
!python code/prepare_data.py --max-prompt-tokens 2048

# Verify splits
import json
with open("data/split_manifest.json") as f:
    m = json.load(f)
print(json.dumps({k: m[k] for k in ["n_patients", "n_rows", "n_trials"]}, indent=2))

In [ ]:
# @title 4. LoRA Fine-tuning (Qwen2.5-1.5B-Instruct, fp16, 3 epochs)
!python code/finetune_lora.py \
    --base-model Qwen/Qwen2.5-1.5B-Instruct \
    --epochs 3 \
    --batch-size 4 \
    --grad-accum 4 \
    --lr 2e-4 \
    --r 16 \
    --alpha 32 \
    --dropout 0.05 \
    --max-length 2048 \
    --warmup-ratio 0.03 \
    --weight-decay 0.01 \
    --dtype auto \
    --grad-checkpoint \
    --out adapters/qwen25-1.5b-lora

In [ ]:
# @title 5. Verify training outputs
import json
with open("results/training_log.json") as f:
    log = json.load(f)
print(json.dumps(log, indent=2))

print("\nAdapter files:")
!ls -la adapters/qwen25-1.5b-lora/

In [ ]:
# @title 6. Before/After evaluation on validation split
!python code/evaluate.py \
    --split val \
    --compare adapters/qwen25-1.5b-lora \
    --batch-size 8 \
    --max-new-tokens 64

In [ ]:
# @title 7. Package adapter for download (zip)
!cd adapters && zip -r qwen25-1.5b-lora.zip qwen25-1.5b-lora

# Download to local machine
from google.colab import files
files.download("adapters/qwen25-1.5b-lora.zip")

## Next Steps (after download)

1. Unzip `qwen25-1.5b-lora.zip` into your local `adapters/` folder
2. Commit the adapter to your GitHub repo (or host on HF)
3. Run local evaluation on test split (when ready):
   ```bash
   python code/evaluate.py --split test --allow-test --compare adapters/qwen25-1.5b-lora --cpu
   ```
4. Continue with Tasks 2-6 using the fine-tuned adapter

**Important**: The test split must remain untouched until final evaluation (Task 1 complete).